In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# =========================================================
# Helpers
# =========================================================

def print_state_labeled(sv, title, tol=1e-10):
    print(f"\n=== {title} ===")
    n = int(np.log2(len(sv.data)))
    for i, amp in enumerate(sv.data):
        if abs(amp) > tol:
            bits = format(i, f"0{n}b")
            print(f"{amp: .4g} |{bits}>")

def deutsch_jozsa_circuit(n, oracle_type="constant"):
    """
    n = number of input qubits
    oracle_type:
        - "constant"
        - "balanced"
    Total qubits = n input + 1 ancilla
    """
    qc = QuantumCircuit(n + 1, n)

    # -----------------------------------------------------
    # 1) Prepare ancilla in |1>
    # -----------------------------------------------------
    qc.x(n)

    # -----------------------------------------------------
    # 2) Hadamard on all qubits
    # -----------------------------------------------------
    qc.h(range(n + 1))

    qc.barrier()

    # -----------------------------------------------------
    # 3) Oracle U_f
    # -----------------------------------------------------
    if oracle_type == "constant":
        # f(x)=0, do nothing
        pass

    elif oracle_type == "balanced":
        # Example balanced oracle: f(x)=x0 xor x1 xor ... xor x_{n-1}
        # Implemented by CNOTs from each input qubit to ancilla
        # for q in range(n):
        #   qc.cx(q, n)    
        qc.cx(0, n)
    else:
        raise ValueError("oracle_type must be 'constant' or 'balanced'")

    qc.barrier()

    # -----------------------------------------------------
    # 4) Hadamard on input register only
    # -----------------------------------------------------
    qc.h(range(n))

    qc.barrier()

    # -----------------------------------------------------
    # 5) Measure only input qubits
    # -----------------------------------------------------
    qc.measure(range(n), range(n))

    return qc


def deutsch_jozsa_state_steps(n, oracle_type="constant"):
    """
    Build the circuit step by step and show the statevector after each stage.
    """
    qc = QuantumCircuit(n + 1)

    # Initial state
    sv0 = Statevector.from_instruction(qc)

    # Step 1: ancilla to |1>
    qc.x(n)
    sv1 = Statevector.from_instruction(qc)

    # Step 2: H on all qubits
    qc.h(range(n + 1))
    sv2 = Statevector.from_instruction(qc)

    # Step 3: oracle
    if oracle_type == "constant":
        pass
    elif oracle_type == "balanced":
        # for q in range(n):
        #   qc.cx(q, n)    
        qc.cx(0, n)
    else:
        raise ValueError("oracle_type must be 'constant' or 'balanced'")

    sv3 = Statevector.from_instruction(qc)

    # Step 4: H on input register
    qc.h(range(n))
    sv4 = Statevector.from_instruction(qc)

    return sv0, sv1, sv2, sv3, sv4


# =========================================================
# Demo
# =========================================================

n = 2

for oracle_type in ["constant", "balanced"]:
    print("\n" + "=" * 60)
    print(f"DEUTSCH-JOZSA DEMO | oracle = {oracle_type}")
    print("=" * 60)

    sv0, sv1, sv2, sv3, sv4 = deutsch_jozsa_state_steps(n, oracle_type)

    print_state_labeled(sv0, "Initial state |000>")
    print_state_labeled(sv1, "After putting ancilla in |1>")
    print_state_labeled(sv2, "After H on all qubits")
    print_state_labeled(sv3, "After oracle (phase information inserted)")
    print_state_labeled(sv4, "After final H on input register (interference result)")

    qc = deutsch_jozsa_circuit(n, oracle_type)
    print("\nCircuit:")
    print(qc.draw("text"))